<a href="https://colab.research.google.com/github/dimi-kri/ML-Ergasia4/blob/main/Ergasia_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Εργασία 4 – Bayes


## Βιβλιοθήκες

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

seed = 0

## Ερώτημα 1 – Φόρτωση δεδομένων και διαχωρισμός σε train-test


In [ ]:
df = pd.read_csv('train.csv')
print('Διαστάσεις:', df.shape)
df.head()

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.1, stratify=df['target'], random_state=seed)
print('Train:', train_df.shape)
print('Test :', test_df.shape)

In [ ]:
train_counts = train_df['target'].value_counts().sort_index()
test_counts = test_df['target'].value_counts().sort_index()

x = np.arange(2)
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Πλήθος
axes[0].bar(x - width/2, train_counts, width, label='Train', color='steelblue', edgecolor='black')
axes[0].bar(x + width/2, test_counts, width, label='Test', color='orange', edgecolor='black')
for i in range(2):
    axes[0].text(i - width/2, train_counts[i], train_counts[i], ha='center', va='bottom')
    axes[0].text(i + width/2, test_counts[i], test_counts[i], ha='center', va='bottom')
axes[0].set_title('Πλήθος target ανά σύνολο')
axes[0].set_ylabel('Πλήθος αναρτήσεων')

# Ποσοστό
train_freq = train_df['target'].value_counts(normalize=True).sort_index() * 100
test_freq = test_df['target'].value_counts(normalize=True).sort_index() * 100
axes[1].bar(x - width/2, train_freq, width, label='Train', color='steelblue', edgecolor='black')
axes[1].bar(x + width/2, test_freq, width, label='Test', color='orange', edgecolor='black')
for i in range(2):
    axes[1].text(i - width/2, train_freq[i], f'{train_freq[i]:.1f}%', ha='center', va='bottom')
    axes[1].text(i + width/2, test_freq[i], f'{test_freq[i]:.1f}%', ha='center', va='bottom')
axes[1].set_title('Ποσοστό target ανά σύνολο')
axes[1].set_ylabel('Ποσοστό (%)')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(['0 – Όχι καταστροφή', '1 – Καταστροφή'])
    ax.legend()
plt.tight_layout()
plt.show()

### Συμπέρασμα

Το dataset δεν είναι τελείως ισορροπημένο, αλλά η διαφορά είναι μικρή:
- 57% των tweets δεν μιλάνε για καταστροφή
- 43% των tweets μιλάνε για καταστροφή

Υπάρχουν αρκετά tweets και από τις δύο κατηγορίες, οπότε το μοντέλο μπορεί να μάθει και τις δύο. Δεν χρειάζεται να αλλάξουμε κάτι στα δεδομένα.

Στην αξιολόγηση όμως ένα μοντέλο που απαντάει πάντα «όχι καταστροφή» θα έβρισκε σωστά το 57% των tweets χωρίς να έχει μάθει τίποτα. Γι' αυτό δεν θα κοιτάξουμε μόνο την accuracy αλλά και άλλες μετρικές precision, recall, F1.

Επειδή χρησιμοποιήσαμε `stratify`, τα ποσοστά είναι ίδια στο train και στο test.

## Ερώτημα 2 – Προεπεξεργασία στηλών


In [ ]:
print('Ελλιπείς τιμές στο αρχικό dataset:')
df.isna().sum()

In [ ]:
def preprocess_columns(data):
    data = data.drop(columns=['id', 'location'])
    data = data.fillna('null')
    data['inputs'] = data['keyword'] + ' ' + data['text']
    return data

train_df = preprocess_columns(train_df)
test_df = preprocess_columns(test_df)

print('Ελλιπείς τιμές μετά (train):', train_df.isna().sum().sum())
print('Ελλιπείς τιμές μετά (test) :', test_df.isna().sum().sum())
train_df.head()

## Ερώτημα 3 – Καθαρισμός κειμένων


| Αρ. | Τεχνική | Περιγραφή (1 γραμμή) | Γιατί την επιλέξαμε |
| - | - | - | - |
| 0 | Remove Unicode & Noise | Αφαιρεί emoji, μη-ASCII χαρακτήρες και emoticons (`:)`). | Τα tweets έχουν πολλά emoji και χαλασμένους χαρακτήρες (π.χ. `Ûª`) που δεν δείχνουν καταστροφή. |
| 1 | Replace URLs, Mentions, Hashtags | Links → `url`, @χρήστες → `atUser`, `#λέξη` → `λέξη`. | Κάθε link και χρήστης είναι διαφορετικός και θα γινόταν άχρηστη «λέξη» στο TF-IDF. Το `#flood` σημαίνει το ίδιο με το `flood`. |
| 9 | Lowercase | Μετατρέπει όλα τα γράμματα σε πεζά. | Το `Fire`, το `FIRE` και το `fire` είναι η ίδια λέξη. |
| 3 | Replace Contractions | Αναπτύσσει συντομεύσεις (`can't` → `cannot`). | Ενώνει διαφορετικές γραφές της ίδιας φράσης. |
| 4 | Remove Numbers | Αφαιρεί τους αριθμούς. | Οι αριθμοί (ώρες, ποσά, ημερομηνίες) από μόνοι τους δεν δείχνουν καταστροφή. |
| 5 | Replace Repetitions of Punctuation | Τα `!!!`, `???`, `...` γίνονται μία λέξη-σήμα. | Η έντονη στίξη μπορεί να δείχνει πανικό, οπότε την κρατάμε ως πληροφορία. |
| 7 | Remove Punctuation | Αφαιρεί όλα τα υπόλοιπα σημεία στίξης. | Τα σύμβολα δεν έχουν νόημα για το TF-IDF. |
| 10 | Remove Stopwords | Αφαιρεί πολύ συχνές λέξεις (`the`, `is`, `and`). | Εμφανίζονται σε όλα τα tweets, άρα δεν ξεχωρίζουν τις δύο κατηγορίες. |
| 14 | Lemmatizing | Φέρνει τη λέξη στη βασική της μορφή (`floods` → `flood`). | Μικραίνει το λεξιλόγιο. Προτιμήθηκε από το stemming γιατί δίνει πραγματικές λέξεις. |




### Συναρτήσεις

In [ ]:
# Πηγή: https://github.com/Deffro/text-preprocessing-techniques (techniques.py)

def removeUnicode(text):
    # Removes unicode strings like "," and "x96"
    text = re.sub(r'(\\u[0-9A-Fa-f]+)', r'', text)
    text = re.sub(r'[^\x00-\x7f]', r'', text)
    return text

def replaceURL(text):
    # Replaces url address with "url"
    text = re.sub(r'((www\.[^\s]+)|(https?://[^\s]+))', 'url', text)
    text = re.sub(r'#([^\s]+)', r'\1', text)
    return text

def replaceAtUser(text):
    # Replaces "@user" with "atUser"
    text = re.sub(r'@[^\s]+', 'atUser', text)
    return text

def removeHashtagInFrontOfWord(text):
    # Removes hastag in front of a word
    text = re.sub(r'#([^\s]+)', r'\1', text)
    return text

def removeNumbers(text):
    # Removes integers
    text = ''.join([i for i in text if not i.isdigit()])
    return text

def replaceMultiExclamationMark(text):
    # Replaces repetitions of exlamation marks
    text = re.sub(r"(\!)\1+", ' multiExclamation ', text)
    return text

def replaceMultiQuestionMark(text):
    # Replaces repetitions of question marks
    text = re.sub(r"(\?)\1+", ' multiQuestion ', text)
    return text

def replaceMultiStopMark(text):
    # Replaces repetitions of stop marks
    text = re.sub(r"(\.)\1+", ' multiStop ', text)
    return text

contraction_patterns = [(r'won\'t', 'will not'), (r'can\'t', 'cannot'), (r'i\'m', 'i am'), (r'ain\'t', 'is not'),
                        (r'(\w+)\'ll', r'\g<1> will'), (r'(\w+)n\'t', r'\g<1> not'), (r'(\w+)\'ve', r'\g<1> have'),
                        (r'(\w+)\'s', r'\g<1> is'), (r'(\w+)\'re', r'\g<1> are'), (r'(\w+)\'d', r'\g<1> would'),
                        (r'&', 'and'), (r'dammit', 'damn it'), (r'dont', 'do not'), (r'wont', 'will not')]

def replaceContraction(text):
    patterns = [(re.compile(regex), repl) for (regex, repl) in contraction_patterns]
    for (pattern, repl) in patterns:
        (text, count) = re.subn(pattern, repl, text)
    return text

def removeEmoticons(text):
    # Removes emoticons from text
    text = re.sub(r':\)|;\)|:-\)|\(-:|:-D|=D|:P|xD|X-p|\^\^|:-*|\^\.\^|\^\-\^|\^\_\^|\,-\)|\)-:|:\'\(|:\(|:-\(|:\S|T\.T|\.\_\.|:<|:-\S|:-<|\*\-\*|:O|=O|=\-O|O\.o|XO|O\_O|:-\@|=/|:/|X\-\(|>\.<|>=\(|D:', '', text)
    return text

### Η συνάρτηση καθαρισμού


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = removeUnicode(text)                               # 0
    text = replaceURL(text)                                  # 1
    text = replaceAtUser(text)                               # 1
    text = removeHashtagInFrontOfWord(text)                  # 1
    text = removeEmoticons(text)                             # 0
    text = text.lower()                                      # 9
    text = replaceContraction(text)                          # 3
    text = removeNumbers(text)                               # 4
    text = replaceMultiExclamationMark(text)                 # 5
    text = replaceMultiQuestionMark(text)                    # 5
    text = replaceMultiStopMark(text)                        # 5
    text = re.sub(r'[^\w\s]', ' ', text)                     # 7
    words = [w for w in text.split() if w not in stop_words] # 10
    words = [lemmatizer.lemmatize(w) for w in words]         # 14
    return ' '.join(words)

train_df['inputs'] = train_df['inputs'].apply(clean_text)
test_df['inputs'] = test_df['inputs'].apply(clean_text)

### Πριν και μετά τον καθαρισμό

In [ ]:
pd.set_option('display.max_colwidth', None)
before_after = pd.DataFrame({
    'Πριν': (df.loc[train_df.index[:6], 'keyword'].fillna('null') + ' ' + df.loc[train_df.index[:6], 'text']).values,
    'Μετά': train_df['inputs'][:6].values
})
before_after

### Συμπέρασμα

Μετά τον καθαρισμό, τα tweets έχουν μόνο πεζά γράμματα και λέξεις με νόημα. Τα links έγιναν url, οι χρήστες atuser, και τα emoji, οι αριθμοί, τα σύμβολα και οι λέξεις όπως the και is αφαιρέθηκαν.

Έτσι το λεξιλόγιο μίκρυνε πολύ. Με λιγότερες και πιο «καθαρές» λέξεις, κάθε λέξη εμφανίζεται περισσότερες φορές, οπότε το μοντέλο μπορεί να μάθει ευκολότερα ποιες λέξεις δείχνουν καταστροφή (π.χ. fire, flood, evacuation).


## Ερώτημα 4 – Διαχωρισμός σε inputs και targets

In [ ]:
x_train = train_df['inputs'].to_numpy()
y_train = train_df['target'].to_numpy()
x_test = test_df['inputs'].to_numpy()
y_test = test_df['target'].to_numpy()

print('x_train:', x_train.shape, ' y_train:', y_train.shape)
print('x_test :', x_test.shape, ' y_test :', y_test.shape)
print('\nΠαράδειγμα:', x_train[0], '→', y_train[0])

## Ερώτημα 5 – TF-IDF


In [ ]:
vectorizer = TfidfVectorizer()
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

print('Μέγεθος λεξιλογίου:', len(vectorizer.vocabulary_))
print('x_train_tfidf:', x_train_tfidf.shape)
print('x_test_tfidf :', x_test_tfidf.shape)
print(f'Ποσοστό μη-μηδενικών τιμών: {100 * x_train_tfidf.nnz / np.prod(x_train_tfidf.shape):.3f}%')

In [ ]:
words = vectorizer.get_feature_names_out()
row = x_train_tfidf[0].toarray().ravel()
top = row.argsort()[::-1][:8]
print('Ανάρτηση:', x_train[0], '\n')
pd.DataFrame({'Λέξη': words[top], 'TF-IDF': row[top].round(3)})

### Συμπέρασμα

Το λεξιλόγιο του train έχει 12.248 διαφορετικές λέξεις, οπότε κάθε ανάρτηση έγινε ένα διάνυσμα με 12.248 αριθμούς. Το train και το test έχουν τον ίδιο αριθμό στηλών, γιατί το test μετατράπηκε με το λεξιλόγιο του train.

Μόνο το 0.077% των τιμών δεν είναι 0, γιατί κάθε tweet έχει λίγες λέξεις από τις 12.248. Ο πίνακας είναι δηλαδή αραιός.

Στο παράδειγμα, τη μεγαλύτερη τιμή την έχει η λέξη threat (0.531). Ακολουθούν σπάνιες λέξεις όπως wedgie και fest. Η λέξη url δεν είναι καν στις 8 πρώτες, γιατί υπάρχει σε πάρα πολλά tweets και άρα παίρνει μικρό IDF. Άρα το TF-IDF δίνει βάρος στις λέξεις που χαρακτηρίζουν τη συγκεκριμένη ανάρτηση.

## Ερώτημα 6 – Gaussian Naive Bayes


In [ ]:
x_train_dense = x_train_tfidf.toarray()
x_test_dense = x_test_tfidf.toarray()

gnb = GaussianNB()
gnb.fit(x_train_dense, y_train)

y_pred_train = gnb.predict(x_train_dense)
y_pred = gnb.predict(x_test_dense)

def get_metrics(y_true, y_hat):
    return {
        'Accuracy': accuracy_score(y_true, y_hat),
        'Precision': precision_score(y_true, y_hat),
        'Recall': recall_score(y_true, y_hat),
        'F1': f1_score(y_true, y_hat)
    }

gnb_results = pd.DataFrame([get_metrics(y_train, y_pred_train), get_metrics(y_test, y_pred)],
                           index=['Train', 'Test']).round(4)
gnb_results

In [ ]:
print(classification_report(y_test, y_pred, target_names=['0 – Όχι καταστροφή', '1 – Καταστροφή']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['Όχι καταστροφή', 'Καταστροφή']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix – Gaussian NB (Test)')
plt.tight_layout()
plt.show()

### Σχολιασμός αποτελεσμάτων

- Accuracy στο test ≈ 0.64. Είναι καλύτερη από ένα μοντέλο που λέει πάντα «όχι καταστροφή» (0.57), αλλά όχι πολύ.
- Recall ≈ 0.84: από τις 327 πραγματικές καταστροφές του test, το μοντέλο βρήκε τις 276 και έχασε τις 51.
- Precision ≈ 0.55: έβγαλε 226 ψευδείς συναγερμούς, δηλαδή σχεδόν οι μισοί συναγερμοί του ήταν λάθος.
- Overfitting: η accuracy στο train 0.86 είναι πολύ μεγαλύτερη από το test 0.64. Το μοντέλο «έμαθε απ' έξω» το train και δεν γενικεύει καλά σε νέα tweets.

Τι σημαίνει αυτό στην πράξη: το σύστημα θα στέλνει μήνυμα 112 στα κινητά κάθε φορά που το μοντέλο προβλέπει «καταστροφή». Υπάρχουν δύο ειδών λάθη:

- Το μοντέλο λέει «δεν είναι καταστροφή» ενώ είναι (False Negative). Αυτό είναι πολύ επικίνδυνο, γιατί δεν θα σταλεί μήνυμα και οι πολίτες δεν θα ειδοποιηθούν.
- Το μοντέλο λέει «καταστροφή» ενώ δεν είναι (False Positive). Αυτό στέλνει ένα άσκοπο μήνυμα. Είναι ενοχλητικό αλλά λιγότερο επικίνδυνο.

Το μοντέλο κάνει λίγα επικίνδυνα λάθη (χάνει 51 καταστροφές) αλλά πολλά άσκοπα μηνύματα (226). Αν οι πολίτες λαμβάνουν συνέχεια λάθος μηνύματα, θα σταματήσουν να τα προσέχουν. Γι' αυτό το μοντέλο χρειάζεται βελτίωση.

Ο Gaussian NB περιμένει οι τιμές κάθε λέξης να είναι «μοιρασμένες» γύρω από μια μέση τιμή. Στο TF-IDF όμως κάθε λέξη έχει τιμή 0 σχεδόν σε όλα τα tweets και μόνο σε λίγα παίρνει κάποια τιμή. Τα δεδομένα δεν μοιάζουν με αυτό που περιμένει το μοντέλο, οπότε κάνει πολλά λάθη.